# EMNIST Balanced CNN Baseline

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader

from datasets.emnist import EMNISTDataset
from models.cnn import SimpleCNN
from utils.label import get_char

---

# 2. Paths

In [ ]:
model_path = Path("../results/simple_cnn.pth")

train_dataset_path = Path(
    "../data/raw/emnist_balenced/emnist-balanced-train.csv"
)


---

# 3. Tạo Dataset

Dataset class có tác dụng:
- Đọc file CSV
- Reshape data về 28x28 (1D - 2D)
- Đổi hướng của EMNIST data (bản raw bị flipped và mirrored)
- Chuẩn hóa giá trị pixel (0-255 -> 0-1)
- Đổi kiểu giữ liệu thành PyTorch tensors


In [ ]:
train_dataset = EMNISTDataset(train_dataset_path)


---

# 4. Tạo DataLoader

DataLoader có tác dụng:

- Chia dữ liệu thành các lô
- Xáo trộn các mẫu
- Cung cấp các lô dữ liệu cho model CNN khi huấn luyện


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)


---

# 5. Quan sát một mẫu ngẫu nhiên


In [ ]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)


Dự đoán đầu ra:

```text
images -> [64, 1, 28, 28]
labels -> [64]
```

---

# 6. Trực quan hóa một mẫu ngẫu nhiên


In [ ]:
plt.imshow(images[0].squeeze(), cmap="gray")
plt.title(get_char(labels[0].item()))
plt.axis("off")
plt.show()


Trực quan hóa giúp xác nhận:

- Hướng hình ảnh
- Pipeline trước xử lí
- Độ chính xác của nhãn

---

# 7. Tạo CNN Model


In [ ]:
model = SimpleCNN()


Cấu trúc model:

```text
Ảnh đầu vào (Kích thước 28x28 pixel (grayscale). Dữ liệu thô từ EMNISTDataset)
→ Conv2D (Tích chập lần 1, dùng 32 bộ lọc quét qua ảnh để tìm các đặc điểm cơ bản như cạnh, góc, nét gạch)
→ ReLU (Loại bỏ các giá trị âm, giúp mô hình học được các quan hệ phi tuyến tính phức tạp)
→ MaxPool (Cực đại hóa, giảm kích thước ảnh xuống một nửa (28x28 -> 14x14), giữ lại những pixel có giá trị lớn nhất (đặc điểm mạnh nhất), giúp giảm khối lượng tính toán và tránh quá tải)
→ Conv2D (Tích chập lần 2, quét các đặc điểm phức tạp hơn, tăng số lượng channel lên 64)
→ ReLU
→ MaxPool (Nén dữ liệu một lần nữa (14x14 -> 7x7), làm cho mô hình nhận diện linh hoạt hơn, không phụ thuộc vào việc chữ cái nằm chính xác ở vị trí nào trong khung hình)
→ Flatten ("Trải phẳng" khối dữ liệu 3D (64x7x7) thành một hàng ngang duy nhất, CNN -> Linear)
→ Linear (Đây là mạng thần kinh truyền thống (Fully Connected), kết hợp tất cả các đặc điểm đã tìm thấy ở trên để bắt đầu suy luận)
→ ReLU
→ Linear (Lớp đầu ra, dùng để tính toán điểm số cho các lớp (Class Scores))
→ Class Scores (Kết quả, trả về 47 con số. Con số nào lớn nhất thì vị trí đó chính là ký tự mà AI dự đoán)
```

---

# 8. Load weight đã lưu trước đó (Optional)

Nếu trước đó đã train một model thì load:


In [ ]:
if model_path.exists():

    model.load_state_dict(
        torch.load(model_path)
    )

    print("Loaded existing model")

else:
    print("Training from scratch")


---

# 9. Forward Pass

Đưa thử một lô dữ liệu (batch) đi qua mạng CNN.


In [ ]:
outputs = model(images)

print(outputs.shape)


Kết quả mong đợi:

```text
[64, 47]
```

Giải thích:
64: Là kích thước của lô dữ liệu (Batch size). 64 hình ảnh đang được đưa vào mô hình cùng một lúc.
47: Tương ứng với số lượng lớp (classes) trong bộ dữ liệu EMNIST Balanced.

---

# 10. Dự đoán


In [ ]:
predictions = outputs.argmax(dim=1)

print(
    "Prediction:",
    get_char(predictions[0].item())
)

print(
    "Actual:",
    get_char(labels[0].item())
)


Ở giai đoạn này, mô hình sẽ dự đoán ngẫu nhiên vì chưa được huấn luyện.

---

# 11. Loss Function


In [ ]:
criterion = nn.CrossEntropyLoss()


CrossEntropyLoss compares:

- predicted logits
- ground-truth labels

and computes prediction error.

---

# 12. Optimizer


In [ ]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


The optimizer updates CNN weights using gradients computed during backpropagation.

---

# 13. Training Loop


In [ ]:
best_loss = float("inf")

for epoch in range(5):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if total_loss < best_loss:

        best_loss = total_loss

        torch.save(
            model.state_dict(),
            "../results/best_model.pth"
        )

    print(f"Epoch {epoch}: {total_loss}")


---

# 14. What Happens During Training

For each batch:

```text
images
→ CNN forward pass
→ predictions
→ loss computation
→ backpropagation
→ weight updates
```

This process gradually reduces prediction error.

---

# 15. Model Saving


In [ ]:
torch.save(
    model.state_dict(),
    "../results/best_model.pth"
)


This stores the learned CNN weights.

Without saving:

- weights remain only in RAM
- restarting the notebook resets the model

---


